<a href="https://colab.research.google.com/github/TomerRippin/Machine-Learning/blob/master/DeepLearning_Regulazation_And_Custom_Dropout_Layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Q1

In [6]:
import torch
import torch.nn as nn
import math

### A, B. Initialize the builder

Note that in the parameter initialization I chose to implement He initialization. I read about this iniitalization online and understood its the best method for linear layers with ReLU activation functions as it prevents vanishing/exploding gradients by scaling weight variance based on input size `fan_in`, keeping signal variance constant through layers.



In [7]:
class SplitLinear(nn.Module):
  def __init__(self, M):
    super(SplitLinear, self).__init__()
    # Check for bad input size during initialization
    assert M % 2 == 0, "Input feature dimension M must be an even number."

    self.M = M

    # Define the shared linear layer and ReLU activation in the constructor
    # This ensures they are initialized once and shared across both halves.
    self.linear_layer = nn.Linear(in_features=int(M/2), out_features=int(M/2))
    self.relu_activation = nn.ReLU()

    # initialize the parameters of the linear layer
    self.init_parameters()

  def init_parameters(self):
    """
    Initialize the parameters in the linear layer using the He method.
    He initialization method is the recommended initialization method
      for deep neural networks that have ReLU to help against gradient decay.
    """
    fan_in = self.M // 2
    limit = math.sqrt(6 / fan_in)
    with torch.no_grad():
          self.linear_layer.weight.uniform_(-limit, limit)
          self.linear_layer.bias.fill_(0)

  def forward(self, X):
    """
    X is NxM tensor. we will split it in the M dimension.
    Aggregate it through a Linear unit, then ReLU.
    return Y = NxM tensor.
    """
    N = X.size(0) # Get the batch size
    M = self.linear_layer.in_features * 2 # M as defined during initialization

    print(f"--- SplitLinear Forward Pass ---")
    print(f"1. Input tensor X (N={N}, M={M}):\n{X}")

    t1, t2 = torch.split(X, int(M/2), dim=1)
    print(f"2. Split X into t1 (N={N}, M/2={int(M/2)}) and t2 (N={N}, M/2={int(M/2)}):")
    print(f"   t1:\n{t1}")
    print(f"   t2:\n{t2}")

    # Apply linear transformation
    t1_linear = self.linear_layer(t1)
    t2_linear = self.linear_layer(t2)
    print(f"3. t1 after Linear layer:\n{t1_linear}")
    print(f"   t2 after Linear layer:\n{t2_linear}")

    # Apply ReLU activation
    t1_relu = self.relu_activation(t1_linear)
    t2_relu = self.relu_activation(t2_linear)
    print(f"   t1 after ReLU activation:\n{t1_relu}")
    print(f"   t2 after ReLU activation:\n{t2_relu}")

    # Concatenate the halves
    Y = torch.cat((t1_relu, t2_relu), dim=1)
    print(f"4. Concatenated output Y (N={N}, M={M}):\n{Y}")
    print(f"--- End SplitLinear Forward Pass ---")

    return Y

### C. Layer Initialization, Example Input, and Printing Intermediate Steps

Here, we will define the input feature dimension `M`, initialize an instance of our custom `SplitLinear` layer, and then create a sample input tensor. This sample input will be passed through the `SplitLinear` layer, and the `forward` method's print statements will show each intermediate step as the data flows through the layer.

In [8]:
M_features = 8

# Initialize the SplitLinear layer
print(f"Initializing SplitLinear layer with M={M_features}")
split_linear_layer = SplitLinear(M=M_features)

# Create a sample input tensor
batch_size = 2
sample_input = torch.randn(batch_size, M_features)
print(f"\nSample Input tensor:\n{sample_input}")

# Pass the sample input through the layer to observe intermediate steps
print(f"\nPassing sample input through SplitLinear layer...")
output = split_linear_layer(sample_input)

print(f"\nFinal output of SplitLinear layer:\n{output}")

Initializing SplitLinear layer with M=8

Sample Input tensor:
tensor([[ 0.2081, -0.2196, -1.5080, -1.7683,  0.6267,  0.8362,  0.1538, -0.4067],
        [-0.3745, -1.2345, -1.5661, -1.2210, -0.6306, -0.8211, -1.2706,  0.4019]])

Passing sample input through SplitLinear layer...
--- SplitLinear Forward Pass ---
1. Input tensor X (N=2, M=8):
tensor([[ 0.2081, -0.2196, -1.5080, -1.7683,  0.6267,  0.8362,  0.1538, -0.4067],
        [-0.3745, -1.2345, -1.5661, -1.2210, -0.6306, -0.8211, -1.2706,  0.4019]])
2. Split X into t1 (N=2, M/2=4) and t2 (N=2, M/2=4):
   t1:
tensor([[ 0.2081, -0.2196, -1.5080, -1.7683],
        [-0.3745, -1.2345, -1.5661, -1.2210]])
   t2:
tensor([[ 0.6267,  0.8362,  0.1538, -0.4067],
        [-0.6306, -0.8211, -1.2706,  0.4019]])
3. t1 after Linear layer:
tensor([[-0.1058, -3.8649,  0.5508,  3.3814],
        [-0.3032, -4.5046,  1.1233,  3.3645]], grad_fn=<AddmmBackward0>)
   t2 after Linear layer:
tensor([[ 0.2796,  0.7159, -0.3793,  0.0076],
        [-0.9694, -1.970

### D. Schematic Diagram of the Layer

The schematic diagram of the `SplitLinear` layer in block form will look like this:

```
+-------------+
|   Input X   |
|   (N x M)   |
+------+------+
       |
       V
+------+------+
|   Split     |
|   (M/2)     |
+---+-----+---+
    |          |
    V          V
+---------+  +---------+
| t1      |  | t2      |
|(N x M/2)|  | (N x M/2)|
+---------+--+---------+
|(Shared Linear Layer) |
+---------+--+---------+
           |
           V
+----------------------+
| Linear (M/2->M/2)    |
| (Weights: W, Bias: b)|
+----------------------+
           |
           V
+-----------------+
| ReLU Activation |
+-----------------+
           |
           V
+---------+  +---------+
| t1'     |  | t2'     |
|(N x M/2)|  |(N x M/2)|
+---------+--+---------+
           |
           V
+--------------------------+
| Concatenate (along M dim)|
+--------------------------+
           |
           V
+-------------+
|   Output Y  |
|   (N x M)   |
+-------------+
```

**Diagram Explanation:**

1.  **Input X (N x M)**: The tensor entering the layer, with N samples and M features.
2.  **Split (M/2)**: The step of splitting the input into two equal halves (t1 and t2) along the feature dimension (dim=1).
3.  **Linear (M/2->M/2) (Shared Layer)**: A single linear layer (identical for both halves) that receives M/2 input features and outputs M/2 features.This layer has its own weights (W) and biases (b).
4.  **ReLU Activation (Shared Layer)**: A single ReLU activation function (identical for both halves) applied to the output of the linear layer.
5.  **Concatenate (along M dim)**: The two processed halves (t1' and t2') are concatenated back along the feature dimension to form the output tensor Y.
6.  **Output Y (N x M)**: The tensor exiting the layer, with the same size as the original input.

### E. Parameter Count Calculation and Comparison

Let's calculate the number of parameters for our `SplitLinear` layer and compare it to a standard `nn.Linear` layer that would handle the same input and output dimensions.

**SplitLinear Parameters:**
The `SplitLinear` layer uses a single shared `nn.Linear` layer. If the full input `X` has `M` features, each split half (`t1` and `t2`) will have `M/2` features. Therefore, the shared linear layer maps `M/2` input features to `M/2` output features.
*   Number of weights = `input_features * output_features` = `(M/2) * (M/2)`
*   Number of biases = `output_features` = `M/2`

total parameters count:
$ \frac{M^2}{4} + \frac{M}{2} $

**Standard `nn.Linear` Parameters:**
A regular `nn.Linear` layer mapping `N x M` input to `N x M` output (i.e., preserving the feature dimension) would have:
*   Number of weights = `input_features * output_features` = `M * M`
*   Number of biases = `output_features` = `M`

total parameters count:
$ M^2 + M $


Calculate the parameters count and difference for our example layer:

SplitLinear(8) = $ \frac{8^2}{4} + \frac{8}{2} = 16 + 4 = 20 $

nn.Linear(8) = $ {8^2} + {8} = 64 + 8 = 72 $

### F. Gradient calculation

To calculate the gradients, we first define the operations occurring in the forward pass.
Let: $X \in \mathbb{R}^{N \times M}$ be the input.

$X_1, X_2 \in \mathbb{R}^{N \times \frac{M}{2}}$ be the two halves of the input.

$W \in \mathbb{R}^{\frac{M}{2} \times \frac{M}{2}}$ and $b \in \mathbb{R}^{\frac{M}{2}}$ be the shared weights and bias.

The intermediate outputs (before activation) are:
$$Z_1 = X_1 W^T + b$$$$Z_2 = X_2 W^T + b$$The final output is $Y = [\text{ReLU}(Z_1), \text{ReLU}(Z_2)]$.

Let $\delta$ be the gradient at the linear output:
$$\delta = \frac{\partial C}{\partial Z} = \frac{\partial C}{\partial Y} \odot \mathbb{1}_{Z>0}$$(Where $\odot$ is the element-wise product and $\mathbb{1}_{Z>0}$  is the indicator function that is $1$ if the input was positive and $0$ otherwise).

We split this gradient into two halves corresponding to our splits: $\delta_1$ and $\delta_2$.

Note that the same $W$ and $b$ are used twice. According to the Multivariable Chain Rule, if a parameter affects the output through multiple paths, its total gradient is the sum of the gradients from each path.

Summing the contributions from both paths gives us the final analytical gradients:

For the Weights ($W$):
$$\frac{\partial C}{\partial W} = \delta_1^T X_1 + \delta_2^T X_2$$

For the Bias ($b$):
$$\frac{\partial C}{\partial b} = \sum_{i=1}^{N} (\delta_{1,i} + \delta_{2,i})$$

### G - What if we split to 4 parts instead of 2?

Because the weights $W$ and bias $b$ are now shared across four separate operations, the Chain Rule dictates that we must sum the gradients from all four paths.

For Weights ($W$):
$$\frac{\partial C}{\partial W} = \delta_1^T X_1 + \delta_2^T X_2 + \delta_3^T X_3 + \delta_4^T X_4$$
For Bias ($b$):
$$\frac{\partial C}{\partial b} = \sum_{k=1}^{4} \sum_{i=1}^{N} \delta_{k,i}$$2.

In addition, we can see that since the input $M$ is now divided by 4, the internal layer becomes smaller. The weight matrix $W$ would be of size $(\frac{M}{4} \times \frac{M}{4})$. Consequently, the gradients $\frac{\partial C}{\partial W}$ would also be smaller matrices, though they would be "denser" in terms of how much information from the original input they represent.

# Q2

### A, B, C - DropNorm Implementation

We will choose at random half of the featurs and zero them.
The rest of the featurs will be nirmelized by the formula:
$$ \hat x_i = \frac {x_i - \mu}{\sqrt{\sigma^2 + c}} $$
While `c` is a a small constant.

The output of the layer will be:
$$ y_i = \gamma_i \hat x_i + \beta_i$$
As $\gamma_i$ and $\beta_i$ are learned parameters of the layer for each feature.


In [12]:
class DropNorm(nn.Module):
  def __init__(self, M_features, epsilon=1e-5):
    super(DropNorm, self).__init__()
    self.M = M_features
    self.epsilon = epsilon

    self.gamma = nn.Parameter(torch.ones(self.M))
    self.beta = nn.Parameter(torch.zeros(self.M))

  def forward(self, X):
    # Prepare mask for broadcasting across all dimensions except the last.
    mask_expanded_shape = (1,) * (X.dim() - 1) + (self.M,)

    if self.training:
      # Randomly select half of the features to drop.
      mask = torch.ones(self.M, device=X.device, dtype=X.dtype)
      indices_to_drop = torch.randperm(self.M, device=X.device)[:self.M // 2]
      mask[indices_to_drop] = 0

      # Features at 'indices_to_drop' become zero.
      mask_expanded = mask.view(mask_expanded_shape)
      X_masked = X * mask_expanded

      # Calculate mean (mu) and variance (sigma^2) for each sample,
      #    considering ONLY the unmasked (active) features.
      M_kept = self.M // 2 # Number of features kept (not zeroed)

      # Sum of active elements for each sample. `X_masked` already has zeros for dropped features.
      sum_active = X_masked.sum(dim=-1, keepdim=True)
      mu = sum_active / M_kept # Mean calculated only over active features

      # Calculate variance of active elements.
      # Subtract mu only from active features, then square and sum.
      diff_from_mu_active = (X_masked - mu) * mask_expanded
      sigma2 = (diff_from_mu_active ** 2).sum(dim=-1, keepdim=True) / M_kept

      # Normalize the kept features using the calculated mu and sigma^2.
      X_hat = (X_masked - mu) / torch.sqrt(sigma2 + self.epsilon)

      # Apply learnable gamma and beta. The features that were dropped must remain 0.
      # Therefore, apply gamma and beta, then re-apply the mask.
      Y_temp = self.gamma.view(mask_expanded_shape) * X_hat + self.beta.view(mask_expanded_shape)
      Y = Y_temp * mask_expanded

      return Y

    else:
      # Calculate mean (mu) and variance (sigma^2) for each sample, over ALL features.
      mu = X.mean(dim=-1, keepdim=True)
      sigma2 = X.var(dim=-1, keepdim=True, unbiased=False)

      # Normalize all features.
      X_hat = (X - mu) / torch.sqrt(sigma2 + self.epsilon)

      # Apply learnable gamma and beta to all normalized features.
      Y = self.gamma.view(mask_expanded_shape) * X_hat + self.beta.view(mask_expanded_shape)

      return Y

### `DropNorm` Layer Implementation Details

**Initialization (`__init__`)**

*   The constructor takes `M_features` (the number of features in the last dimension of the input tensor) and an `epsilon` value (a small constant to prevent division by zero during normalization).
*   It initializes two learnable parameters: `gamma` and `beta`.
    *   `gamma` (for scaling) is initialized to `torch.ones(self.M)`.
    *   `beta` (for shifting) is initialized to `torch.zeros(self.M)`.
* This setup ensures that at the beginning of training, the `DropNorm` layer behaves somewhat like an identity function, This contributes to more stable training.


### D - Train a deep network using built-in PyTorch layers

First, we'll load the MNIST-Fashion dataset and define the necessary data transformations. Then, we'll create data loaders for training and testing.

In [13]:
# Import necessary libraries for dataset loading and transformations
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Define data transformations
# We'll convert images to tensors.
transform = transforms.Compose([
    transforms.ToTensor() # Convert images to tensors
])

# Load the training dataset
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)

# Load the test dataset
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

# Define data loaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Batch size: {batch_size}")

100%|██████████| 26.4M/26.4M [00:01<00:00, 21.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 345kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 6.36MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.9MB/s]

Training dataset size: 60000
Test dataset size: 10000
Batch size: 64


### Define the Neural Network Model with Built-in Layers

Now, we'll define a simple feedforward neural network. Since FashionMNIST images are 28x28 pixels, we'll flatten them into a 784-dimensional vector. We'll include built-in `nn.BatchNorm1d` and `nn.Dropout` layers.

In [14]:
import torch.nn as nn
import torch.nn.functional as F

class NetWithBuiltInLayers(nn.Module):
    def __init__(self):
        super(NetWithBuiltInLayers, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(0.25)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(0.25)
        self.fc3 = nn.Linear(256, 10) # 10 classes for FashionMNIST

    def forward(self, x):
        # Flatten the image
        x = x.view(-1, 28 * 28)

        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        return x

# Instantiate the model
model_builtin = NetWithBuiltInLayers()
print(model_builtin)


NetWithBuiltInLayers(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.25, inplace=False)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.25, inplace=False)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
)


### Training and Evaluation Loop for `NetWithBuiltInLayers`

Now, we'll implement the training and evaluation functions. We will use `Adam` optimizer and `CrossEntropyLoss` for our classification task as they are Industry standarts. The training loop will iterate through the dataset for 10 epochs as it looks like it pretty much converge afterwards.

In [15]:
import torch.optim as optim

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_builtin.to(device)

# Hyperparameters
learning_rate = 0.001
num_epochs = 10

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_builtin.parameters(), lr=learning_rate)

# Training function
def train(model, train_loader, criterion, optimizer, device):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = 100 * correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

# Evaluation function
def evaluate(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    correct_predictions = 0
    total_samples = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    accuracy = 100 * correct_predictions / total_samples
    return accuracy

print(f"Training on: {device}")

# Train the model
for epoch in range(num_epochs):
    train_loss, train_accuracy = train(model_builtin, train_loader, criterion, optimizer, device)
    test_accuracy = evaluate(model_builtin, test_loader, device)
    print(f'Epoch [{epoch+1}/{num_epochs}], ' \
          f'Loss: {train_loss:.4f}, ' \
          f'Train Accuracy: {train_accuracy:.2f}%, ' \
          f'Test Accuracy: {test_accuracy:.2f}%')


Training on: cpu
Epoch [1/10], Loss: 0.4739, Train Accuracy: 83.04%, Test Accuracy: 84.82%
Epoch [2/10], Loss: 0.3694, Train Accuracy: 86.53%, Test Accuracy: 85.77%
Epoch [3/10], Loss: 0.3364, Train Accuracy: 87.60%, Test Accuracy: 87.95%
Epoch [4/10], Loss: 0.3136, Train Accuracy: 88.41%, Test Accuracy: 88.08%
Epoch [5/10], Loss: 0.2981, Train Accuracy: 88.92%, Test Accuracy: 88.03%
Epoch [6/10], Loss: 0.2863, Train Accuracy: 89.28%, Test Accuracy: 88.37%
Epoch [7/10], Loss: 0.2742, Train Accuracy: 89.81%, Test Accuracy: 88.69%
Epoch [8/10], Loss: 0.2604, Train Accuracy: 90.19%, Test Accuracy: 88.55%
Epoch [9/10], Loss: 0.2532, Train Accuracy: 90.41%, Test Accuracy: 88.33%
Epoch [10/10], Loss: 0.2438, Train Accuracy: 90.89%, Test Accuracy: 88.57%


### E. Define the Neural Network Model with Custom `DropNorm` Layers

Now, we'll define a new feedforward neural network, `NetWithDropNormLayers`, which will replace `nn.BatchNorm1d` and `nn.Dropout` with our custom `DropNorm` layer. The input to the `DropNorm` layer will be the output dimension of the preceding `nn.Linear` layer.

In [16]:
class NetWithDropNormLayers(nn.Module):
    def __init__(self):
        super(NetWithDropNormLayers, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.dropnorm1 = DropNorm(M_features=512)
        self.fc2 = nn.Linear(512, 256)
        self.dropnorm2 = DropNorm(M_features=256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        # Flatten the image
        x = x.view(-1, 28 * 28)

        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropnorm1(x)

        x = self.fc2(x)
        x = F.relu(x)
        x = self.dropnorm2(x)

        x = self.fc3(x)
        return x

# Instantiate the model
model_dropnorm = NetWithDropNormLayers()
print(model_dropnorm)

NetWithDropNormLayers(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (dropnorm1): DropNorm()
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (dropnorm2): DropNorm()
  (fc3): Linear(in_features=256, out_features=10, bias=True)
)


### Training and Evaluation Loop for `NetWithDropNormLayers`

We will now train the `NetWithDropNormLayers` model using the training hyper-parameters This will allow us to compare its performance against the model using built-in layers.

In [17]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dropnorm.to(device)

# Hyperparameters (using the same as before for fair comparison)
learning_rate = 0.001
num_epochs = 10

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer_dropnorm = optim.Adam(model_dropnorm.parameters(), lr=learning_rate) # New optimizer for the new model

print(f"Training on: {device}")

# Train the model
for epoch in range(num_epochs):
    train_loss_dropnorm, train_accuracy_dropnorm = train(model_dropnorm, train_loader, criterion, optimizer_dropnorm, device)
    test_accuracy_dropnorm = evaluate(model_dropnorm, test_loader, device)
    print(f'Epoch [{epoch+1}/{num_epochs}], ' \
          f'DropNorm Loss: {train_loss_dropnorm:.4f}, ' \
          f'DropNorm Train Accuracy: {train_accuracy_dropnorm:.2f}%, ' \
          f'DropNorm Test Accuracy: {test_accuracy_dropnorm:.2f}%')


Training on: cpu
Epoch [1/10], DropNorm Loss: 0.6391, DropNorm Train Accuracy: 77.14%, DropNorm Test Accuracy: 81.80%
Epoch [2/10], DropNorm Loss: 0.4753, DropNorm Train Accuracy: 82.77%, DropNorm Test Accuracy: 84.59%
Epoch [3/10], DropNorm Loss: 0.4328, DropNorm Train Accuracy: 84.38%, DropNorm Test Accuracy: 85.01%
Epoch [4/10], DropNorm Loss: 0.4055, DropNorm Train Accuracy: 85.23%, DropNorm Test Accuracy: 86.29%
Epoch [5/10], DropNorm Loss: 0.3887, DropNorm Train Accuracy: 85.90%, DropNorm Test Accuracy: 85.77%
Epoch [6/10], DropNorm Loss: 0.3747, DropNorm Train Accuracy: 86.23%, DropNorm Test Accuracy: 87.05%
Epoch [7/10], DropNorm Loss: 0.3658, DropNorm Train Accuracy: 86.59%, DropNorm Test Accuracy: 86.41%
Epoch [8/10], DropNorm Loss: 0.3536, DropNorm Train Accuracy: 87.01%, DropNorm Test Accuracy: 87.40%
Epoch [9/10], DropNorm Loss: 0.3473, DropNorm Train Accuracy: 87.38%, DropNorm Test Accuracy: 86.98%
Epoch [10/10], DropNorm Loss: 0.3379, DropNorm Train Accuracy: 87.68%, Dro

### Model Performance Comparison

**`NetWithBuiltInLayers` Test Accuracy:** 88.57%

**`NetWithDropNormLayers` Test Accuracy:** 87.71%

From these results, the `NetWithBuiltInLayers` model performed slightly better, achieving a test accuracy of 88.57% compared to `NetWithDropNormLayers` at 87.71%. The difference is pretty small might even be neglible. nevertheless I will try to explain the differences between the networks and why we should prefer one over the other.

**Explanation for the Difference:**

The slight performance difference can be attributed to the fundamental differences in how `DropNorm` operates compared to the standard `nn.BatchNorm1d` and `nn.Dropout` layers:

1. `DropNorm` implements an aggressive regularization strategy by consistently zeroing out half of the features for *all samples within a batch* during training. While `nn.Dropout` also zeros out features, it does so randomly and independently for *each element* in the input, providing more diverse training examples. The consistent dropping in `DropNorm` might force the model to learn from a more restricted set of features at any given moment, potentially hindering its ability to leverage all available information optimally for this particular task.

2. `DropNorm` calculates mean and variance based *only on the active (non-dropped) features* for each sample. In contrast, `nn.BatchNorm1d` normalizes using statistics from *all features across the entire batch* (and uses learned running statistics during inference).

3.  The combination of `nn.BatchNorm1d` (which helps with faster convergence and some regularization) and `nn.Dropout` (which prevents co-adaptation of features) in `NetWithBuiltInLayers` appears to be slightly more effective for FashionMNIST under these training conditions than the `DropNorm` mechanism.

For those reasons, we will prefer to use the `NetWithBuiltInLayers`.

### F. Backpropagation for DropNorm Layer

To calculate the gradient of the cost function $C$ (which can also be denoted as $L$ for Loss) with respect to the input $x_i$ (where $\frac{\partial C}{\partial x_i}$ is the gradient flowing back), we must account for the normalization process and the feature masking.

#### 1. Definitions and Setup

Let $S$ be the set of indices of the features that were not zeroed out during the forward pass. Based on the definition, $|S| = \frac{D}{2}$ (where $D$ is the total number of features, which is `M_features` in our implementation). For any $i \notin S$, the output $y_i = 0$, so $\frac{\partial C}{\partial x_i} = 0$.

For $i \in S$, the transformation is:

$$y_i = \gamma_i \hat{x}_i + \beta_i, \quad \text{where} \quad \hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

The mean $\mu$ and variance $\sigma^2$ are calculated over the active features $S$:

$$\mu = \frac{1}{|S|} \sum_{j \in S} x_j, \quad \sigma^2 = \frac{1}{|S|} \sum_{j \in S} (x_j - \mu)^2$$

#### 2. Chain Rule Expansion

Using the chain rule for the normalization step, the gradient for $i \in S$ is:

$$\frac{\partial C}{\partial x_i} = \frac{\partial C}{\partial \hat{x}_i} \cdot \frac{\partial \hat{x}_i}{\partial x_i} + \frac{\partial C}{\partial \mu} \cdot \frac{\partial \mu}{\partial x_i} + \frac{\partial C}{\partial \sigma^2} \cdot \frac{\partial \sigma^2}{\partial x_i}$$

First, note that $\frac{\partial C}{\partial \hat{x}_i} = \frac{\partial C}{\partial y_i} \cdot \gamma_i$.

#### 3. Internal Derivatives

The Components First, we find how the internal statistics change with respect to the input $x_i$:

Mean Derivative: $\mu = \frac{1}{|S|} \sum_{j \in S} x_j \implies \frac{\partial \mu}{\partial x_i} = \frac{1}{|S|}$

Variance Derivative: $\sigma^2 = \frac{1}{|S|} \sum_{j \in S} (x_j - \mu)^2 \implies \frac{\partial \sigma^2}{\partial x_i} = \frac{2(x_i - \mu)}{|S|}$

Normalized Value (Direct): $\hat{x}_i = (x_i - \mu) \cdot V^{-1/2} \implies \frac{\partial \hat{x}_i}{\partial x_i} = \frac{1}{\sqrt{V}}$

#### 4. Gradients of Intermediate Variables ($\frac{\partial C}{\partial \sigma^2}$ and $\frac{\partial C}{\partial \mu}$)

Now we calculate how sensitive the Loss is to the changes in the batch statistics:
A. For the Variance ($\sigma^2$):
The variance affects all $\hat{x}_j$ in the set via the denominator.

$$\frac{\partial C}{\partial \sigma^2} = \sum_{j \in S} \frac{\partial C}{\partial \hat{x}_j} \cdot \frac{\partial \hat{x}_j}{\partial \sigma^2} = \sum_{j \in S} \frac{\partial C}{\partial \hat{x}_j} \cdot (x_j - \mu) \cdot \left( -\frac{1}{2} V^{-3/2} \right)$$

B. For the Mean ($\mu$):

The mean affects all $\hat{x}_j$ through both the numerator ($x_j - \mu$) and the variance. However, it can be simplified to:

$$\frac{\partial C}{\partial \mu} = \sum_{j \in S} \frac{\partial C}{\partial \hat{x}_j} \cdot \frac{\partial \hat{x}_j}{\partial \mu} = \sum_{j \in S} \frac{\partial C}{\partial \hat{x}_j} \cdot \left( -\frac{1}{\sqrt{V}} \right)$$

#### 5. Putting it All Together
Now, we substitute these components back into the main chain rule equation from the start:
$$\frac{\partial C}{\partial x_i} = \underbrace{\frac{\partial C}{\partial \hat{x}_i} \cdot \frac{1}{\sqrt{V}}}_{\text{Direct Path}} + \underbrace{\frac{\partial C}{\partial \mu} \cdot \frac{1}{|S|}}_{\text{Path via Mean}} + \underbrace{\frac{\partial C}{\partial \sigma^2} \cdot \frac{2(x_i - \mu)}{|S|}}_{\text{Path via Variance}}$$

Substituting the sums we found in Step 2:

$$\frac{\partial C}{\partial x_i} = \frac{\frac{\partial C}{\partial \hat{x}_i}}{\sqrt{V}} + \left[ \sum \frac{\partial C}{\partial \hat{x}_j} \frac{-1}{\sqrt{V}} \right] \frac{1}{|S|} + \left[ \sum \frac{\partial C}{\partial \hat{x}_j} (x_j - \mu) \frac{-1}{2 V \sqrt{V}} \right] \frac{2(x_i - \mu)}{|S|}$$

#### 6. Algebraic Simplification
To reach the final efficient form, we factor out $\frac{1}{|S| \sqrt{V}}$:

The first term becomes $|S| \frac{\partial C}{\partial \hat{x}_i}$.

The second term becomes $-\sum \frac{\partial C}{\partial \hat{x}_j}$.

In the third term, the $2$ and $1/2$ cancel out. We notice that $\frac{x_j - \mu}{\sqrt{V}}$ is exactly $\hat{x}_j$. This simplifies the term to $-\hat{x}_i \sum (\frac{\partial C}{\partial \hat{x}_j} \cdot \hat{x}_j)$.The Final Result:

$$\frac{\partial C}{\partial x_i} = \frac{1}{|S| \sqrt{\sigma^2 + \epsilon}} \left[ |S| \frac{\partial C}{\partial \hat{x}_i} - \sum_{j \in S} \frac{\partial C}{\partial \hat{x}_j} - \hat{x}_i \sum_{j \in S} \left( \frac{\partial C}{\partial \hat{x}_j} \hat{x}_j \right) \right]$$

**Summary for all $i$:**

$$\frac{\partial C}{\partial x_i} =\begin{cases}\frac{\gamma_i}{|S| \sqrt{\sigma^2 + \epsilon}} \left[ |S| \frac{\partial C}{\partial y_i} - \sum_{j \in S} \frac{\partial C}{\partial y_j} \gamma_j - \hat{x}_i \sum_{j \in S} \left( \frac{\partial C}{\partial y_j} \gamma_j \hat{x}_j \right) \right] & i \in S \\0 & i \notin S\end{cases}$$